# Z 偏置—频率标定

每个 Z 点先测 XY 关闭基准，再扫描驱动 IF。只有完整运行可分析；确认峰和扫描边界后手动发布配置。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "execute":
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in Platform_LQ or its execute directory")
sys.path.insert(0, str(ROOT))
import numpy as np
from src.zfreq import ZSettings, run_calibration, analyze
from src.config import load_pi, save_hardware, publish_z
BACKEND_NAME = "YOUR_BACKEND"
QUBITS = [9]
Z_VALUES = np.linspace(-.20, .20, 9)
FREQUENCY_HALF_WIDTH_GHZ = .20
FREQUENCY_POINTS = 11
SHOTS = 500
SETTINGS = ZSettings()
RUN_HARDWARE = False
RUN_MODE = "new"
RESUME_RUN_TAG = ""
PUBLISH_CONFIG = False


In [ ]:
run = None
if RUN_HARDWARE:
    from src.progress import watch_run
    with watch_run(ROOT / "result" / "zfreq", len(QUBITS) * len(Z_VALUES) * (FREQUENCY_POINTS + 1),
                   run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG,
                   title='Z 频率网格标定'):
        from lqcloud import LQCloudProvider
        backend = LQCloudProvider().get_backend(BACKEND_NAME)
        qconf = {int(q["id"]): q for q in backend.config["properties"]["qubit_metrics"]}
        pi_cal = load_pi(ROOT / "config", BACKEND_NAME, QUBITS)
        frequencies = {
            q: np.linspace(qconf[q]["df_10"] - FREQUENCY_HALF_WIDTH_GHZ,
                           qconf[q]["df_10"] + FREQUENCY_HALF_WIDTH_GHZ,
                           FREQUENCY_POINTS) for q in QUBITS}
        print("Planned jobs:", len(QUBITS) * len(Z_VALUES) * (FREQUENCY_POINTS + 1))
        save_hardware(ROOT / "config", backend)
        run = run_calibration(backend, qconf, pi_cal, Z_VALUES, frequencies,
                              ROOT / "result" / "zfreq", SETTINGS, shots=SHOTS,
                              run_mode=RUN_MODE, resume_run_tag=RESUME_RUN_TAG)
        print("Saved:", run)


In [ ]:
# To inspect a completed earlier run, assign its absolute path to run first.
if run is not None:
    from IPython.display import display
    import pandas as pd
    display(pd.DataFrame(analyze(run)))
if PUBLISH_CONFIG:
    if run is None:
        raise ValueError("Select a completed run before publishing")
    print("Published:", publish_z(run, ROOT / "config", BACKEND_NAME))
